# Prepare three-task VoxVietnam protocols from Kaggle Input

This notebook reads the existing sharded Parquet dataset from
`/kaggle/input/voxvietnam-dataset`. It scans metadata first, performs
streaming audio quality control, selects leakage-safe protocols, and
writes only the compact derived dataset to `/kaggle/working`.

Output protocol:

- 230 closed-set speakers, 30 audio each: 20 train / 5 validation / 5 test.
- 50 validation speakers, 15 audio each.
- 50 official-test speakers, 15 audio each.
- Validation/test: 5 enrollment audio per speaker; remaining audio are queries.
- One positive and five negative trials per query.
- 25 known-gallery and 25 unknown speakers per open-set split.
- Speaker-disjoint train/validation/test and hard 12 GiB audio limit.

The full redundant `train` split is inventoried but never decoded.
`train_small` supplies train/validation speakers; official `test`
supplies held-out test speakers. Keep output private unless gated
redistribution terms explicitly permit publication.


## 1. Install dependencies


In [ ]:
!pip install -q "datasets[audio]" soundfile scipy pyarrow tqdm


## 2. Input discovery and configuration


In [ ]:
import json
import shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input/voxvietnam-dataset")
OUTPUT_ROOT = Path(
    "/kaggle/working/voxvietnam_ecapa_three_task_v1"
)
QC_SAMPLE_PATH = Path("/kaggle/working/voxvietnam_qc_sample.csv")
QC_INVENTORY_PATH = Path("/kaggle/working/voxvietnam_qc_inventory.csv")
SELECTION_PLAN_PATH = Path(
    "/kaggle/working/voxvietnam_selection_plan.csv"
)

# Reserve 230 speakers with 30 valid audio for training. Validation
# speakers are different identities and need only 15 valid audio.
TRAIN_SPEAKERS = 230
VALIDATION_SPEAKERS = 50
TEST_SPEAKERS = 50
TRAIN_AUDIO_PER_SPEAKER = 30
EVALUATION_AUDIO_PER_SPEAKER = 15
ENROLLMENT_AUDIO_PER_SPEAKER = 5
NEGATIVE_TRIALS_PER_QUERY = 5
MAX_BYTES = 12 * 1024**3
CLOSED_TRAIN_AUDIO = 20
CLOSED_VALIDATION_AUDIO = 5
OPEN_SET_KNOWN_SPEAKERS = 25
MIN_DURATION_SEC = 2.0
MAX_DURATION_SEC = 10.0
SILENCE_TOP_DB = 35.0
TRIM_PAD_SEC = 0.10
QC_SAMPLE_PER_PARTITION = 250
SEED = 42

assert INPUT_ROOT.is_dir(), f"Missing Kaggle input: {INPUT_ROOT}"
assert not OUTPUT_ROOT.exists(), (
    f"Refusing to overwrite existing output: {OUTPUT_ROOT}"
)
disk = shutil.disk_usage("/kaggle/working")
print(f"Working disk free: {disk.free / 1024**3:.2f} GiB")
assert disk.free > MAX_BYTES + 4 * 1024**3, (
    "Need at least 4 GiB working headroom beyond audio budget"
)


## 3. Self-contained streaming builder


In [ ]:
"""Build compact VoxVietnam protocols for three ECAPA speaker tasks.

The builder streams gated Hugging Face data, writes 16 kHz PCM16 WAV files,
keeps protocol roles disjoint where required, and emits closed-set
identification, verification, and open-set identification ground truth. It
intentionally refuses to overwrite output or exceed the byte budget.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import shutil
from collections import Counter
from itertools import chain
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping

import numpy as np
import soundfile as sf

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


HF_DATASET = "hustep-lab/VoxVietnam-Dataset"
TARGET_SAMPLE_RATE = 16_000
DEFAULT_MAX_BYTES = 12 * 1024**3

METADATA_FIELDS = (
    "audio_id",
    "audio_path",
    "speaker_id",
    "normalized_speaker_id",
    "split",
    "split_name",
    "split_role",
    "protocol",
    "role",
    "duration_sec",
    "sample_rate",
    "num_channels",
    "checksum",
    "source_dataset",
    "source_partition",
    "source_index",
)

TRIAL_FIELDS = (
    "trial_id",
    "enrollment_speaker_id",
    "query_audio_path",
    "query_speaker_id",
    "label",
)

PROTOCOL_AUDIO_FIELDS = ("audio_path", "speaker_id", "checksum")
OPEN_QUERY_FIELDS = (
    "query_audio_path",
    "query_speaker_id",
    "is_known",
    "checksum",
)


def _write_csv(path: Path, fields: tuple[str, ...], rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def _stable_key(seed: int, value: str) -> str:
    return hashlib.sha256(f"{seed}:{value}".encode("utf-8")).hexdigest()


def select_speaker_splits(
    training_counts: Mapping[str, int],
    test_counts: Mapping[str, int],
    *,
    train_speakers: int = 600,
    validation_speakers: int = 75,
    test_speakers: int = 75,
    train_audio_per_speaker: int = 30,
    evaluation_audio_per_speaker: int = 15,
    seed: int = 42,
) -> dict[str, list[str]]:
    """Select deterministic speakers from official train and test partitions."""

    numeric = (
        train_speakers,
        validation_speakers,
        test_speakers,
        train_audio_per_speaker,
        evaluation_audio_per_speaker,
    )
    if any(value < 1 for value in numeric):
        raise ValueError("Speaker and audio counts must be positive")

    source_overlap = set(training_counts) & set(test_counts)
    if source_overlap:
        raise ValueError(
            f"Official source partitions overlap by {len(source_overlap)} speakers"
        )

    training_candidates = [
        speaker
        for speaker, count in training_counts.items()
        if count >= train_audio_per_speaker
    ]
    training_candidates.sort(key=lambda speaker: _stable_key(seed, speaker))
    if len(training_candidates) < train_speakers:
        raise ValueError(
            f"Need {train_speakers} train speakers with at least "
            f"{train_audio_per_speaker} audio; found {len(training_candidates)}"
        )
    selected_train = training_candidates[:train_speakers]
    selected_train_set = set(selected_train)

    validation_candidates = [
        speaker
        for speaker, count in training_counts.items()
        if speaker not in selected_train_set
        and count >= evaluation_audio_per_speaker
    ]
    validation_candidates.sort(key=lambda speaker: _stable_key(seed, speaker))
    if len(validation_candidates) < validation_speakers:
        raise ValueError(
            f"Need {validation_speakers} remaining validation speakers with at "
            f"least {evaluation_audio_per_speaker} audio; found "
            f"{len(validation_candidates)} after reserving train speakers"
        )

    test_candidates = [
        speaker
        for speaker, count in test_counts.items()
        if count >= evaluation_audio_per_speaker
    ]
    test_candidates.sort(key=lambda speaker: _stable_key(seed + 1, speaker))
    if len(test_candidates) < test_speakers:
        raise ValueError(
            f"Need {test_speakers} official-test speakers with at least "
            f"{evaluation_audio_per_speaker} audio; found {len(test_candidates)}"
        )

    return {
        "train": selected_train,
        "validation": validation_candidates[:validation_speakers],
        "test": test_candidates[:test_speakers],
    }


def _waveform(audio: Any) -> tuple[np.ndarray, int]:
    if not isinstance(audio, Mapping) or "array" not in audio:
        raise ValueError("VoxVietnam audio must contain array and sampling_rate")
    sample_rate = int(audio.get("sampling_rate", 0))
    if sample_rate != TARGET_SAMPLE_RATE:
        raise ValueError(
            f"Expected {TARGET_SAMPLE_RATE} Hz source audio; received {sample_rate} Hz"
        )
    values = np.asarray(audio["array"], dtype=np.float32)
    if values.ndim == 2:
        axis = 0 if values.shape[0] <= 8 else 1
        values = values.mean(axis=axis)
    if values.ndim != 1 or values.size == 0 or not np.isfinite(values).all():
        raise ValueError("Source audio is empty, non-finite, or not mono-compatible")
    peak = float(np.max(np.abs(values)))
    if peak <= np.finfo(np.float32).eps:
        raise ValueError("Source audio is silent")
    return np.clip(values, -1.0, 1.0), sample_rate


def _verification_trials(
    rows: list[dict[str, Any]],
    *,
    negative_trials_per_query: int,
    seed: int,
) -> list[dict[str, Any]]:
    enrollment_speakers = sorted(
        {row["normalized_speaker_id"] for row in rows if row["role"] == "ENROLLMENT"}
    )
    queries = [row for row in rows if row["role"] == "QUERY"]
    trials: list[dict[str, Any]] = []
    for query in queries:
        query_speaker = query["normalized_speaker_id"]
        targets = [speaker for speaker in enrollment_speakers if speaker != query_speaker]
        targets.sort(
            key=lambda speaker: _stable_key(
                seed, f"{query['audio_id']}:{speaker}"
            )
        )
        selected_negatives = targets[: min(negative_trials_per_query, len(targets))]
        pairs = [(query_speaker, 1), *((speaker, 0) for speaker in selected_negatives)]
        for enrollment_speaker, label in pairs:
            trial_index = len(trials)
            trials.append(
                {
                    "trial_id": f"trial_{trial_index:08d}",
                    "enrollment_speaker_id": enrollment_speaker,
                    "query_audio_path": query["audio_path"],
                    "query_speaker_id": query_speaker,
                    "label": label,
                }
            )
    return trials


def _audio_protocol_rows(rows: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        {
            "audio_path": row["audio_path"],
            "speaker_id": row["normalized_speaker_id"],
            "checksum": row["checksum"],
        }
        for row in rows
    ]


def _protocol_manifest(
    path: Path,
    rows: list[dict[str, Any]],
    relative_path: Path,
) -> dict[str, Any]:
    return {
        "path": relative_path.as_posix(),
        "rows": len(rows),
        "sha256": sha256_file(path),
    }


def materialize_voxvietnam_subset(
    records: Iterable[Mapping[str, Any]],
    speaker_splits: Mapping[str, list[str]],
    *,
    output_root: Path,
    train_audio_per_speaker: int = 30,
    evaluation_audio_per_speaker: int = 15,
    enrollment_audio_per_speaker: int = 5,
    negative_trials_per_query: int = 5,
    closed_set_train_audio_per_speaker: int = 20,
    closed_set_validation_audio_per_speaker: int = 5,
    open_set_known_speakers: int = 25,
    max_bytes: int = DEFAULT_MAX_BYTES,
    seed: int = 42,
) -> dict[str, Any]:
    """Write selected records into current project dataset layout."""

    if output_root.exists():
        raise FileExistsError(f"Refusing to overwrite dataset: {output_root}")
    if not 1 <= enrollment_audio_per_speaker < evaluation_audio_per_speaker:
        raise ValueError(
            "enrollment_audio_per_speaker must be positive and smaller than "
            "evaluation_audio_per_speaker"
        )
    closed_test_audio_per_speaker = (
        train_audio_per_speaker
        - closed_set_train_audio_per_speaker
        - closed_set_validation_audio_per_speaker
    )
    if (
        closed_set_train_audio_per_speaker < 1
        or closed_set_validation_audio_per_speaker < 1
        or closed_test_audio_per_speaker < 1
    ):
        raise ValueError(
            "closed-set partitions must each contain at least one audio per speaker"
        )
    if negative_trials_per_query < 1 or max_bytes < 1:
        raise ValueError("Trial count and byte budget must be positive")

    split_sets = {name: set(values) for name, values in speaker_splits.items()}
    if set(split_sets) != {"train", "validation", "test"}:
        raise ValueError("speaker_splits must contain train/validation/test")
    if any(
        left & right
        for index, left in enumerate(split_sets.values())
        for right in list(split_sets.values())[index + 1 :]
    ):
        raise ValueError("Speaker splits must be pairwise disjoint")
    if not 1 <= open_set_known_speakers < len(split_sets["validation"]):
        raise ValueError(
            "open_set_known_speakers must leave known and unknown validation speakers"
        )
    if open_set_known_speakers >= len(split_sets["test"]):
        raise ValueError(
            "open_set_known_speakers must leave known and unknown test speakers"
        )

    speaker_to_split = {
        speaker: split for split, speakers in split_sets.items() for speaker in speakers
    }
    ordered_speakers = [
        speaker for split in ("train", "validation", "test")
        for speaker in sorted(split_sets[split])
    ]
    normalized = {
        speaker: f"voxvi_spk_{index:04d}"
        for index, speaker in enumerate(ordered_speakers, start=1)
    }
    caps = {
        "train": train_audio_per_speaker,
        "validation": evaluation_audio_per_speaker,
        "test": evaluation_audio_per_speaker,
    }
    selected_counts: Counter[str] = Counter()
    metadata: dict[str, list[dict[str, Any]]] = {
        "train": [],
        "validation": [],
        "test": [],
    }
    total_bytes = 0
    seen_checksums: set[str] = set()
    skipped_duplicate_content = 0

    output_root.mkdir(parents=True)
    try:
        for source_index, record in enumerate(records):
            source_speaker = str(record.get("speaker", "")).strip()
            split = speaker_to_split.get(source_speaker)
            if split is None or selected_counts[source_speaker] >= caps[split]:
                continue

            waveform, sample_rate = _waveform(record.get("audio"))
            normalized_speaker = normalized[source_speaker]
            audio_key = hashlib.sha256(
                f"{source_speaker}:{record.get('_source_partition', '')}:"
                f"{source_index}".encode("utf-8")
            ).hexdigest()[:20]
            audio_id = f"voxvi_{audio_key}"
            relative = (
                Path(split) / "audio" / normalized_speaker / f"{audio_id}.wav"
            )
            destination = output_root / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            sf.write(destination, waveform, sample_rate, subtype="PCM_16")
            checksum = sha256_file(destination)
            if checksum in seen_checksums:
                destination.unlink()
                skipped_duplicate_content += 1
                continue
            seen_checksums.add(checksum)
            file_bytes = destination.stat().st_size
            total_bytes += file_bytes
            if total_bytes > max_bytes:
                raise ValueError(
                    f"Dataset byte budget exceeded: {total_bytes} > {max_bytes}"
                )

            speaker_position = selected_counts[source_speaker]
            if split == "train":
                role = "TRAIN"
                protocol = "AAM_TRAIN"
            else:
                role = (
                    "ENROLLMENT"
                    if speaker_position < enrollment_audio_per_speaker
                    else "QUERY"
                )
                protocol = "COSINE_VERIFICATION"
            duration = waveform.size / sample_rate
            metadata[split].append(
                {
                    "audio_id": audio_id,
                    "audio_path": relative.as_posix(),
                    "speaker_id": source_speaker,
                    "normalized_speaker_id": normalized_speaker,
                    "split": split,
                    "split_name": f"voxvietnam_{split}",
                    "split_role": role,
                    "protocol": protocol,
                    "role": role,
                    "duration_sec": f"{duration:.6f}",
                    "sample_rate": sample_rate,
                    "num_channels": 1,
                    "checksum": checksum,
                    "source_dataset": HF_DATASET,
                    "source_partition": record.get("_source_partition", ""),
                    "source_index": source_index,
                }
            )
            selected_counts[source_speaker] += 1

            if all(
                selected_counts[speaker] >= caps[speaker_to_split[speaker]]
                for speaker in speaker_to_split
            ):
                break

        missing = {
            speaker: caps[split] - selected_counts[speaker]
            for speaker, split in speaker_to_split.items()
            if selected_counts[speaker] < caps[split]
        }
        if missing:
            preview = dict(list(sorted(missing.items()))[:5])
            raise ValueError(
                f"Source ended before selected speakers reached audio caps: {preview}"
            )

        split_manifest: dict[str, Any] = {}
        for split, rows in metadata.items():
            rows.sort(key=lambda row: (row["normalized_speaker_id"], row["audio_id"]))
            metadata_path = output_root / split / "metadata.csv"
            _write_csv(metadata_path, METADATA_FIELDS, rows)
            details: dict[str, Any] = {
                "audio": len(rows),
                "speakers": len({row["normalized_speaker_id"] for row in rows}),
                "metadata": f"{split}/metadata.csv",
                "metadata_sha256": sha256_file(metadata_path),
            }
            if split in {"validation", "test"}:
                trials = _verification_trials(
                    rows,
                    negative_trials_per_query=negative_trials_per_query,
                    seed=seed + (1 if split == "validation" else 2),
                )
                trials_path = output_root / split / "verification_trials.csv"
                _write_csv(trials_path, TRIAL_FIELDS, trials)
                details.update(
                    {
                        "trials": len(trials),
                        "positive_trials": sum(int(row["label"]) for row in trials),
                        "negative_trials": sum(
                            int(row["label"]) == 0 for row in trials
                        ),
                        "trials_path": f"{split}/verification_trials.csv",
                        "trials_sha256": sha256_file(trials_path),
                    }
                )
            split_manifest[split] = details

        protocols_root = output_root / "protocols"
        protocol_manifest: dict[str, Any] = {
            "closed_set": {}, "verification": {}, "open_set": {}
        }

        train_by_speaker: dict[str, list[dict[str, Any]]] = {}
        for row in metadata["train"]:
            train_by_speaker.setdefault(row["normalized_speaker_id"], []).append(row)
        closed_partitions: dict[str, list[dict[str, Any]]] = {
            "classifier_train.csv": [],
            "validation_queries.csv": [],
            "test_queries.csv": [],
        }
        for speaker in sorted(train_by_speaker):
            rows = sorted(train_by_speaker[speaker], key=lambda row: row["audio_id"])
            train_end = closed_set_train_audio_per_speaker
            valid_end = train_end + closed_set_validation_audio_per_speaker
            closed_partitions["classifier_train.csv"].extend(rows[:train_end])
            closed_partitions["validation_queries.csv"].extend(rows[train_end:valid_end])
            closed_partitions["test_queries.csv"].extend(rows[valid_end:])
        closed_path_sets: list[set[str]] = []
        closed_checksum_sets: list[set[str]] = []
        for filename, selected in closed_partitions.items():
            rows = _audio_protocol_rows(selected)
            relative = Path("protocols") / "closed_set" / filename
            _write_csv(output_root / relative, PROTOCOL_AUDIO_FIELDS, rows)
            protocol_manifest["closed_set"][filename.removesuffix(".csv")] = (
                _protocol_manifest(output_root / relative, rows, relative)
            )
            closed_path_sets.append({row["audio_path"] for row in rows})
            closed_checksum_sets.append({row["checksum"] for row in rows})
        closed_disjoint = all(
            not left & right
            for index, left in enumerate(closed_path_sets)
            for right in closed_path_sets[index + 1 :]
        ) and all(
            not left & right
            for index, left in enumerate(closed_checksum_sets)
            for right in closed_checksum_sets[index + 1 :]
        )
        if not closed_disjoint:
            raise ValueError("Closed-set audio path/checksum leakage detected")

        open_speaker_sets: dict[str, set[str]] = {}
        for split in ("validation", "test"):
            split_rows = metadata[split]
            enrollment = [row for row in split_rows if row["role"] == "ENROLLMENT"]
            verification_trials = _verification_trials(
                split_rows,
                negative_trials_per_query=negative_trials_per_query,
                seed=seed + (1 if split == "validation" else 2),
            )
            enrollment_relative = Path("protocols") / "verification" / f"{split}_enrollment.csv"
            trials_relative = Path("protocols") / "verification" / f"{split}_trials.csv"
            enrollment_rows = _audio_protocol_rows(enrollment)
            _write_csv(output_root / enrollment_relative, PROTOCOL_AUDIO_FIELDS, enrollment_rows)
            _write_csv(output_root / trials_relative, TRIAL_FIELDS, verification_trials)
            protocol_manifest["verification"][f"{split}_enrollment"] = _protocol_manifest(
                output_root / enrollment_relative, enrollment_rows, enrollment_relative
            )
            protocol_manifest["verification"][f"{split}_trials"] = _protocol_manifest(
                output_root / trials_relative, verification_trials, trials_relative
            )

            speakers = sorted(
                {row["normalized_speaker_id"] for row in split_rows},
                key=lambda value: _stable_key(seed + (10 if split == "validation" else 20), value),
            )
            known_speakers = set(speakers[:open_set_known_speakers])
            open_speaker_sets[split] = set(speakers)
            gallery_selected = [
                row for row in enrollment
                if row["normalized_speaker_id"] in known_speakers
            ]
            query_selected = [
                row for row in split_rows if row["role"] == "QUERY"
            ]
            gallery_rows = _audio_protocol_rows(gallery_selected)
            query_rows = [
                {
                    "query_audio_path": row["audio_path"],
                    "query_speaker_id": row["normalized_speaker_id"],
                    "is_known": int(row["normalized_speaker_id"] in known_speakers),
                    "checksum": row["checksum"],
                }
                for row in query_selected
            ]
            gallery_relative = Path("protocols") / "open_set" / f"{split}_gallery.csv"
            queries_relative = Path("protocols") / "open_set" / f"{split}_queries.csv"
            _write_csv(output_root / gallery_relative, PROTOCOL_AUDIO_FIELDS, gallery_rows)
            _write_csv(output_root / queries_relative, OPEN_QUERY_FIELDS, query_rows)
            protocol_manifest["open_set"][f"{split}_gallery"] = _protocol_manifest(
                output_root / gallery_relative, gallery_rows, gallery_relative
            )
            protocol_manifest["open_set"][f"{split}_queries"] = _protocol_manifest(
                output_root / queries_relative, query_rows, queries_relative
            )
        if open_speaker_sets["validation"] & open_speaker_sets["test"]:
            raise ValueError("Open-set validation/test speaker leakage detected")

        mapping_rows = [
            {
                "speaker_id": speaker,
                "normalized_speaker_id": normalized[speaker],
                "split": speaker_to_split[speaker],
            }
            for speaker in ordered_speakers
        ]
        _write_csv(
            output_root / "speaker_mapping.csv",
            ("speaker_id", "normalized_speaker_id", "split"),
            mapping_rows,
        )

        manifest = {
            "dataset": "voxvietnam_ecapa_three_task_v1",
            "source_dataset": HF_DATASET,
            "license": "CC-BY-NC-4.0; gated source terms also apply",
            "seed": seed,
            "audio_format": "WAV PCM16 mono 16 kHz",
            "total_audio": sum(len(rows) for rows in metadata.values()),
            "total_audio_bytes": total_bytes,
            "maximum_audio_bytes": max_bytes,
            "skipped_duplicate_content": skipped_duplicate_content,
            "train_ground_truth": "normalized_speaker_id class",
            "verification_ground_truth": "binary trial label: 1 same, 0 different",
            "closed_set_ground_truth": "speaker_id class shared across classifier train/validation/test",
            "open_set_ground_truth": "query_speaker_id plus is_known membership",
            "enrollment_audio_per_speaker": enrollment_audio_per_speaker,
            "negative_trials_per_query": negative_trials_per_query,
            "splits": split_manifest,
            "protocols": protocol_manifest,
            "protocol_configuration": {
                "closed_set_train_audio_per_speaker": closed_set_train_audio_per_speaker,
                "closed_set_validation_audio_per_speaker": closed_set_validation_audio_per_speaker,
                "closed_set_test_audio_per_speaker": closed_test_audio_per_speaker,
                "open_set_known_speakers_per_split": open_set_known_speakers,
            },
            "invariants": {
                "speaker_disjoint": True,
                "duplicate_checksum": 0,
                "validation_selects_checkpoint_and_threshold": True,
                "test_used_once_for_final_evaluation": True,
                "byte_budget_respected": total_bytes <= max_bytes,
                "closed_set_audio_and_checksum_disjoint": closed_disjoint,
                "open_set_validation_test_speakers_disjoint": True,
                "open_set_unknown_absent_from_gallery": True,
            },
        }
        (output_root / "manifest.json").write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        (output_root / "README.md").write_text(
            "# VoxVietnam ECAPA three-task subset v1\n\n"
            "Compact subset for one ECAPA checkpoint serving closed-set "
            "identification, claimed-centroid verification, and rejected "
            "open-set identification. Ground truth lives under `protocols/`.\n\n"
            "Source: `hustep-lab/VoxVietnam-Dataset`. License: CC BY-NC 4.0. "
            "The source is gated; do not publish this derived package unless the "
            "source access terms permit redistribution.\n",
            encoding="utf-8",
        )
        return manifest
    except Exception:
        shutil.rmtree(output_root)
        raise


def _hf_token() -> str:
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token
    try:
        from huggingface_hub import get_token
    except ImportError:
        get_token = None
    cached = get_token() if get_token is not None else None
    if not cached:
        raise RuntimeError(
            "HF_TOKEN is required. Accept VoxVietnam access terms, then run "
            "`hf auth login` or configure a Kaggle Secret named HF_TOKEN."
        )
    return cached


def _hf_speaker_counts(split: str, token: str) -> Counter[str]:
    try:
        from datasets import load_dataset
    except ImportError as error:
        raise RuntimeError(
            "Hugging Face datasets is required; install project requirements"
        ) from error
    dataset = load_dataset(
        HF_DATASET,
        split=split,
        streaming=True,
        token=token,
    ).select_columns(["speaker"])
    return Counter(str(row["speaker"]) for row in dataset)


def _hf_records(split: str, token: str) -> Iterator[dict[str, Any]]:
    from datasets import load_dataset

    dataset = load_dataset(
        HF_DATASET,
        split=split,
        streaming=True,
        token=token,
    ).select_columns(["audio", "speaker"])
    for row in dataset:
        yield {**row, "_source_partition": split}


def build_from_huggingface(
    *,
    output_root: Path,
    training_source_split: str = "train_small",
    test_source_split: str = "test",
    train_speakers: int = 230,
    validation_speakers: int = 50,
    test_speakers: int = 50,
    train_audio_per_speaker: int = 30,
    evaluation_audio_per_speaker: int = 15,
    enrollment_audio_per_speaker: int = 5,
    negative_trials_per_query: int = 5,
    closed_set_train_audio_per_speaker: int = 20,
    closed_set_validation_audio_per_speaker: int = 5,
    open_set_known_speakers: int = 25,
    max_bytes: int = DEFAULT_MAX_BYTES,
    seed: int = 42,
) -> dict[str, Any]:
    """Scan gated source metadata, select speakers, then stream selected audio."""

    token = _hf_token()
    training_counts = _hf_speaker_counts(training_source_split, token)
    test_counts = _hf_speaker_counts(test_source_split, token)
    splits = select_speaker_splits(
        training_counts,
        test_counts,
        train_speakers=train_speakers,
        validation_speakers=validation_speakers,
        test_speakers=test_speakers,
        train_audio_per_speaker=train_audio_per_speaker,
        evaluation_audio_per_speaker=evaluation_audio_per_speaker,
        seed=seed,
    )
    records = chain(
        _hf_records(training_source_split, token),
        _hf_records(test_source_split, token),
    )
    return materialize_voxvietnam_subset(
        records,
        splits,
        output_root=output_root,
        train_audio_per_speaker=train_audio_per_speaker,
        evaluation_audio_per_speaker=evaluation_audio_per_speaker,
        enrollment_audio_per_speaker=enrollment_audio_per_speaker,
        negative_trials_per_query=negative_trials_per_query,
        closed_set_train_audio_per_speaker=closed_set_train_audio_per_speaker,
        closed_set_validation_audio_per_speaker=closed_set_validation_audio_per_speaker,
        open_set_known_speakers=open_set_known_speakers,
        max_bytes=max_bytes,
        seed=seed,
    )


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--output-root",
        type=Path,
        default=Path("data/datasets/voxvietnam_ecapa_three_task_v1"),
    )
    parser.add_argument("--training-source-split", default="train_small")
    parser.add_argument("--test-source-split", default="test")
    parser.add_argument("--train-speakers", type=int, default=230)
    parser.add_argument("--validation-speakers", type=int, default=50)
    parser.add_argument("--test-speakers", type=int, default=50)
    parser.add_argument("--train-audio-per-speaker", type=int, default=30)
    parser.add_argument("--evaluation-audio-per-speaker", type=int, default=15)
    parser.add_argument("--enrollment-audio-per-speaker", type=int, default=5)
    parser.add_argument("--negative-trials-per-query", type=int, default=5)
    parser.add_argument("--closed-set-train-audio-per-speaker", type=int, default=20)
    parser.add_argument("--closed-set-validation-audio-per-speaker", type=int, default=5)
    parser.add_argument("--open-set-known-speakers", type=int, default=25)
    parser.add_argument("--max-gib", type=float, default=12.0)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    manifest = build_from_huggingface(
        output_root=args.output_root,
        training_source_split=args.training_source_split,
        test_source_split=args.test_source_split,
        train_speakers=args.train_speakers,
        validation_speakers=args.validation_speakers,
        test_speakers=args.test_speakers,
        train_audio_per_speaker=args.train_audio_per_speaker,
        evaluation_audio_per_speaker=args.evaluation_audio_per_speaker,
        enrollment_audio_per_speaker=args.enrollment_audio_per_speaker,
        negative_trials_per_query=args.negative_trials_per_query,
        closed_set_train_audio_per_speaker=args.closed_set_train_audio_per_speaker,
        closed_set_validation_audio_per_speaker=args.closed_set_validation_audio_per_speaker,
        open_set_known_speakers=args.open_set_known_speakers,
        max_bytes=int(args.max_gib * 1024**3),
        seed=args.seed,
    )
    print(json.dumps(manifest, ensure_ascii=False, indent=2))
    return 0


## 4. Discover shards and audit speaker metadata


In [ ]:
from collections import Counter

import pandas as pd
import pyarrow.parquet as pq

SOURCE_FILES = {
    "train_small": sorted(INPUT_ROOT.glob("train_small-*.parquet")),
    "train": sorted(INPUT_ROOT.glob("train-*.parquet")),
    "test": sorted(INPUT_ROOT.glob("test-*.parquet")),
}
assert SOURCE_FILES["train_small"], "Missing train_small Parquet shards"
assert SOURCE_FILES["test"], "Missing test Parquet shards"

source_summary = []
for split, files in SOURCE_FILES.items():
    rows = 0
    size = 0
    for path in files:
        parquet = pq.ParquetFile(path)
        names = set(parquet.schema_arrow.names)
        assert {"audio", "speaker"} <= names, (
            f"{path.name} missing audio/speaker: {sorted(names)}"
        )
        rows += parquet.metadata.num_rows
        size += path.stat().st_size
    source_summary.append({
        "partition": split,
        "shards": len(files),
        "rows": rows,
        "gib": size / 1024**3,
    })
source_summary = pd.DataFrame(source_summary)
display(source_summary)


def parquet_speaker_counts(files):
    counts = Counter()
    missing = 0
    for path in files:
        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(
            columns=["speaker"], batch_size=262_144, use_threads=True
        ):
            index = batch.schema.get_field_index("speaker")
            for value in batch.column(index).to_pylist():
                speaker = "" if value is None else str(value).strip()
                if speaker:
                    counts[speaker] += 1
                else:
                    missing += 1
    return counts, missing


raw_counts = {}
missing_speakers = {}
for split in ("train_small", "test"):
    raw_counts[split], missing_speakers[split] = parquet_speaker_counts(
        SOURCE_FILES[split]
    )

audit_rows = []
for split, counts in raw_counts.items():
    for speaker, audio_count in counts.items():
        audit_rows.append({
            "partition": split,
            "speaker": speaker,
            "audio_count": audio_count,
            "eligible_15": audio_count >= EVALUATION_AUDIO_PER_SPEAKER,
            "eligible_30": audio_count >= TRAIN_AUDIO_PER_SPEAKER,
        })
speaker_audit = pd.DataFrame(audit_rows)
speaker_audit.to_csv(
    "/kaggle/working/voxvietnam_speaker_audit.csv", index=False
)
display(
    speaker_audit.groupby("partition").agg(
        speakers=("speaker", "nunique"),
        eligible_15=("eligible_15", "sum"),
        eligible_30=("eligible_30", "sum"),
        min_audio=("audio_count", "min"),
        median_audio=("audio_count", "median"),
        max_audio=("audio_count", "max"),
    )
)

source_overlap = set(raw_counts["train_small"]) & set(raw_counts["test"])
assert not source_overlap, (
    f"Speaker overlap between train_small/test: {len(source_overlap)}"
)
assert not any(missing_speakers.values()), missing_speakers


## 5. Define streaming preprocessing and QC


In [ ]:
import csv
import io
from math import gcd

import soundfile as sf
from datasets import load_dataset
from scipy.signal import resample_poly
from tqdm.auto import tqdm

QC_FIELDS = (
    "row_id", "source_partition", "source_file", "source_row",
    "speaker", "status", "reason", "original_sample_rate",
    "original_channels", "duration_sec", "content_hash",
)


def source_records(split):
    for path in SOURCE_FILES[split]:
        relative = path.relative_to(INPUT_ROOT).as_posix()
        dataset = load_dataset(
            "parquet",
            data_files={"source": str(path)},
            split="source",
            streaming=True,
        ).select_columns(["audio", "speaker"])
        for row_index, row in enumerate(dataset):
            yield {
                "speaker": "" if row["speaker"] is None
                else str(row["speaker"]).strip(),
                "audio": row["audio"],
                "_source_partition": split,
                "_source_file": relative,
                "_source_row": row_index,
            }


def decode_audio(audio):
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        values = samples.data
        if hasattr(values, "detach"):
            values = values.detach().cpu().numpy()
        return np.asarray(values, dtype=np.float32), int(
            samples.sample_rate
        )

    if isinstance(audio, Mapping) and "array" in audio:
        return (
            np.asarray(audio["array"], dtype=np.float32),
            int(audio.get("sampling_rate", 0)),
        )

    if isinstance(audio, Mapping) and audio.get("bytes") is not None:
        values, sample_rate = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32",
            always_2d=False,
        )
        return np.asarray(values), int(sample_rate)

    if isinstance(audio, Mapping) and audio.get("path"):
        values, sample_rate = sf.read(
            audio["path"], dtype="float32", always_2d=False
        )
        return np.asarray(values), int(sample_rate)

    raise ValueError("Unsupported audio representation")


def preprocess_audio(audio):
    values, original_sample_rate = decode_audio(audio)
    if original_sample_rate <= 0:
        raise ValueError("invalid_sample_rate")
    if values.ndim == 1:
        original_channels = 1
    elif values.ndim == 2:
        original_channels = min(values.shape)
        axis = 0 if values.shape[0] <= 8 else 1
        values = values.mean(axis=axis)
    else:
        raise ValueError("invalid_shape")
    values = np.asarray(values, dtype=np.float32)
    if values.ndim != 1 or values.size == 0:
        raise ValueError("empty_audio")
    if not np.isfinite(values).all():
        raise ValueError("non_finite_audio")

    peak = float(np.max(np.abs(values)))
    if peak <= np.finfo(np.float32).eps:
        raise ValueError("silent_audio")
    threshold = peak * 10 ** (-SILENCE_TOP_DB / 20.0)
    active = np.flatnonzero(np.abs(values) >= threshold)
    if active.size == 0:
        raise ValueError("no_active_audio")
    pad = int(round(TRIM_PAD_SEC * original_sample_rate))
    start = max(0, int(active[0]) - pad)
    stop = min(values.size, int(active[-1]) + pad + 1)
    values = values[start:stop]

    if original_sample_rate != TARGET_SAMPLE_RATE:
        divisor = gcd(original_sample_rate, TARGET_SAMPLE_RATE)
        values = resample_poly(
            values,
            TARGET_SAMPLE_RATE // divisor,
            original_sample_rate // divisor,
        ).astype(np.float32, copy=False)

    minimum = int(round(MIN_DURATION_SEC * TARGET_SAMPLE_RATE))
    maximum = int(round(MAX_DURATION_SEC * TARGET_SAMPLE_RATE))
    if values.size < minimum:
        raise ValueError("duration_below_minimum")
    if values.size > maximum:
        offset = (values.size - maximum) // 2
        values = values[offset : offset + maximum]

    values = np.clip(values, -1.0, 1.0).astype(np.float32, copy=False)
    pcm16 = np.rint(values * 32767.0).astype("<i2", copy=False)
    return {
        "waveform": values,
        "sample_rate": TARGET_SAMPLE_RATE,
        "original_sample_rate": original_sample_rate,
        "original_channels": original_channels,
        "duration_sec": values.size / TARGET_SAMPLE_RATE,
        "content_hash": hashlib.sha256(pcm16.tobytes()).hexdigest(),
    }


# Override embedded builder preprocessing. The materializer resolves this
# global function at runtime and therefore writes the exact QC waveform.
def _waveform(audio):
    processed = preprocess_audio(audio)
    return processed["waveform"], processed["sample_rate"]


def scan_audio_quality(output_path, limit_per_partition=None):
    seen_content = set()
    status_counts = Counter()
    partition_totals = {
        split: sum(
            pq.ParquetFile(path).metadata.num_rows
            for path in SOURCE_FILES[split]
        )
        for split in ("train_small", "test")
    }
    with output_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=QC_FIELDS)
        writer.writeheader()
        for split in ("train_small", "test"):
            total = partition_totals[split]
            if limit_per_partition is not None:
                total = min(total, limit_per_partition)
            progress = tqdm(total=total, desc=f"QC {split}")
            for partition_index, record in enumerate(source_records(split)):
                if (
                    limit_per_partition is not None
                    and partition_index >= limit_per_partition
                ):
                    break
                row_id = hashlib.sha256(
                    f"{split}:{record['_source_file']}:"
                    f"{record['_source_row']}".encode("utf-8")
                ).hexdigest()[:24]
                result = {
                    "row_id": row_id,
                    "source_partition": split,
                    "source_file": record["_source_file"],
                    "source_row": record["_source_row"],
                    "speaker": record["speaker"],
                    "status": "invalid",
                    "reason": "",
                    "original_sample_rate": "",
                    "original_channels": "",
                    "duration_sec": "",
                    "content_hash": "",
                }
                try:
                    if not record["speaker"]:
                        raise ValueError("missing_speaker")
                    processed = preprocess_audio(record["audio"])
                    result.update({
                        "original_sample_rate": processed[
                            "original_sample_rate"
                        ],
                        "original_channels": processed[
                            "original_channels"
                        ],
                        "duration_sec": f"{processed['duration_sec']:.6f}",
                        "content_hash": processed["content_hash"],
                    })
                    if processed["content_hash"] in seen_content:
                        result["status"] = "duplicate"
                        result["reason"] = "duplicate_content"
                    else:
                        seen_content.add(processed["content_hash"])
                        result["status"] = "valid"
                        result["reason"] = "ok"
                except Exception as error:
                    result["reason"] = str(error)
                writer.writerow(result)
                status_counts[(split, result["status"], result["reason"])] += 1
                progress.update(1)
            progress.close()
    return status_counts


## 6. Sample QC before full scan


In [ ]:
sample_status = scan_audio_quality(
    QC_SAMPLE_PATH,
    limit_per_partition=QC_SAMPLE_PER_PARTITION,
)
sample_qc = pd.read_csv(QC_SAMPLE_PATH)
display(
    sample_qc.groupby(
        ["source_partition", "status", "reason"], dropna=False
    ).size().rename("rows").reset_index()
)
display(
    sample_qc.loc[sample_qc["status"] == "valid"].groupby(
        "source_partition"
    ).agg(
        audio=("row_id", "count"),
        median_duration=("duration_sec", "median"),
        min_duration=("duration_sec", "min"),
        max_duration=("duration_sec", "max"),
    )
)
assert (sample_qc["status"] == "valid").any(), "No valid sample audio"


## 7. Full streaming QC inventory

This is the long CPU step. It decodes `train_small` and `test` once,
never loads an entire Parquet shard into RAM, and does not touch full
`train`. Restart from this cell only if the sample QC looks reasonable.


In [ ]:
full_status = scan_audio_quality(QC_INVENTORY_PATH)
quality = pd.read_csv(QC_INVENTORY_PATH)
display(
    quality.groupby(
        ["source_partition", "status", "reason"], dropna=False
    ).size().rename("rows").reset_index()
)

valid_quality = quality.loc[quality["status"] == "valid"].copy()
valid_counts_frame = (
    valid_quality.groupby(["source_partition", "speaker"])
    .size()
    .rename("valid_audio")
    .reset_index()
)
display(
    valid_counts_frame.groupby("source_partition").agg(
        speakers=("speaker", "nunique"),
        eligible_15=(
            "valid_audio",
            lambda values: int((values >= EVALUATION_AUDIO_PER_SPEAKER).sum()),
        ),
        eligible_30=(
            "valid_audio",
            lambda values: int((values >= TRAIN_AUDIO_PER_SPEAKER).sum()),
        ),
        median_valid_audio=("valid_audio", "median"),
    )
)


## 8. Freeze deterministic speaker and audio selection


In [ ]:
quality_counts = {
    split: Counter(dict(
        valid_counts_frame.loc[
            valid_counts_frame["source_partition"] == split,
            ["speaker", "valid_audio"],
        ].itertuples(index=False, name=None)
    ))
    for split in ("train_small", "test")
}
speaker_splits = select_speaker_splits(
    quality_counts["train_small"],
    quality_counts["test"],
    train_speakers=TRAIN_SPEAKERS,
    validation_speakers=VALIDATION_SPEAKERS,
    test_speakers=TEST_SPEAKERS,
    train_audio_per_speaker=TRAIN_AUDIO_PER_SPEAKER,
    evaluation_audio_per_speaker=EVALUATION_AUDIO_PER_SPEAKER,
    seed=SEED,
)

speaker_to_split = {
    speaker: split
    for split, speakers in speaker_splits.items()
    for speaker in speakers
}
caps = {
    "train": TRAIN_AUDIO_PER_SPEAKER,
    "validation": EVALUATION_AUDIO_PER_SPEAKER,
    "test": EVALUATION_AUDIO_PER_SPEAKER,
}
plan_parts = []
for speaker, dataset_split in speaker_to_split.items():
    source_partition = (
        "test" if dataset_split == "test" else "train_small"
    )
    candidates = valid_quality.loc[
        (valid_quality["source_partition"] == source_partition)
        & (valid_quality["speaker"] == speaker)
    ].copy()
    candidates["selection_key"] = candidates["row_id"].map(
        lambda value: _stable_key(SEED, str(value))
    )
    selected = candidates.sort_values("selection_key").head(
        caps[dataset_split]
    )
    assert len(selected) == caps[dataset_split]
    selected["dataset_split"] = dataset_split
    plan_parts.append(selected)

selection_plan = pd.concat(plan_parts, ignore_index=True)
assert selection_plan["row_id"].is_unique
assert selection_plan["content_hash"].is_unique
selection_plan.to_csv(SELECTION_PLAN_PATH, index=False)
display(
    selection_plan.groupby("dataset_split").agg(
        audio=("row_id", "count"),
        speakers=("speaker", "nunique"),
        duration_hours=("duration_sec", lambda x: x.sum() / 3600),
    )
)


## 9. Materialize selected WAV files and create three protocols


In [ ]:
def planned_records(plan):
    for source_file, group in plan.groupby("source_file", sort=True):
        wanted = {
            int(row.source_row): row
            for row in group.itertuples(index=False)
        }
        path = INPUT_ROOT / source_file
        dataset = load_dataset(
            "parquet",
            data_files={"source": str(path)},
            split="source",
            streaming=True,
        ).select_columns(["audio", "speaker"])
        found = 0
        for row_index, row in enumerate(dataset):
            planned = wanted.get(row_index)
            if planned is None:
                continue
            speaker = "" if row["speaker"] is None else str(
                row["speaker"]
            ).strip()
            assert speaker == planned.speaker
            found += 1
            yield {
                "speaker": speaker,
                "audio": row["audio"],
                "_source_partition": planned.source_partition,
                "_source_file": source_file,
                "_source_row": row_index,
            }
        assert found == len(wanted), (
            f"Missing planned rows in {source_file}: "
            f"{found}/{len(wanted)}"
        )


manifest = materialize_voxvietnam_subset(
    planned_records(selection_plan),
    speaker_splits,
    output_root=OUTPUT_ROOT,
    train_audio_per_speaker=TRAIN_AUDIO_PER_SPEAKER,
    evaluation_audio_per_speaker=EVALUATION_AUDIO_PER_SPEAKER,
    enrollment_audio_per_speaker=ENROLLMENT_AUDIO_PER_SPEAKER,
    negative_trials_per_query=NEGATIVE_TRIALS_PER_QUERY,
    max_bytes=MAX_BYTES,
    closed_set_train_audio_per_speaker=CLOSED_TRAIN_AUDIO,
    closed_set_validation_audio_per_speaker=CLOSED_VALIDATION_AUDIO,
    open_set_known_speakers=OPEN_SET_KNOWN_SPEAKERS,
    seed=SEED,
)

shutil.copy2(
    SELECTION_PLAN_PATH,
    OUTPUT_ROOT / "source_selection_plan.csv",
)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 10. Inspect current files and create package metadata/manifest


In [ ]:
import hashlib
import pandas as pd

GENERATED_PACKAGE_FILES = {
    "dataset_metadata.json",
    "file_manifest.csv",
}
required_files = [
    "manifest.json",
    "speaker_mapping.csv",
    "train/metadata.csv",
    "validation/metadata.csv",
    "test/metadata.csv",
    "protocols/closed_set/classifier_train.csv",
    "protocols/closed_set/validation_queries.csv",
    "protocols/closed_set/test_queries.csv",
    "protocols/verification/validation_enrollment.csv",
    "protocols/verification/validation_trials.csv",
    "protocols/verification/test_enrollment.csv",
    "protocols/verification/test_trials.csv",
    "protocols/open_set/validation_gallery.csv",
    "protocols/open_set/validation_queries.csv",
    "protocols/open_set/test_gallery.csv",
    "protocols/open_set/test_queries.csv",
]


def sha256_file_status(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


assert OUTPUT_ROOT.is_dir(), f"Output directory not found: {OUTPUT_ROOT}"

status = pd.DataFrame([
    {
        "path": relative,
        "exists": (OUTPUT_ROOT / relative).is_file(),
        "bytes": (
            (OUTPUT_ROOT / relative).stat().st_size
            if (OUTPUT_ROOT / relative).is_file()
            else 0
        ),
    }
    for relative in required_files
])
display(status)

missing_required = status.loc[~status["exists"], "path"].tolist()
source_manifest_path = OUTPUT_ROOT / "manifest.json"
source_manifest = (
    json.loads(source_manifest_path.read_text(encoding="utf-8"))
    if source_manifest_path.is_file()
    else None
)

audio_checksums = {}
split_summary = []
missing_audio = []
for split in ("train", "validation", "test"):
    metadata_path = OUTPUT_ROOT / split / "metadata.csv"
    if not metadata_path.is_file():
        split_summary.append({
            "split": split,
            "metadata": "missing",
            "audio": 0,
            "speakers": 0,
            "missing_audio": None,
        })
        continue

    frame = pd.read_csv(metadata_path)
    required_columns = {
        "audio_path", "normalized_speaker_id", "checksum"
    }
    absent_columns = required_columns - set(frame.columns)
    if absent_columns:
        raise ValueError(
            f"{metadata_path} missing columns: {sorted(absent_columns)}"
        )

    split_missing = [
        str(relative)
        for relative in frame["audio_path"]
        if not (OUTPUT_ROOT / str(relative)).is_file()
    ]
    missing_audio.extend(split_missing)
    audio_checksums.update({
        str(row.audio_path): str(row.checksum)
        for row in frame.itertuples(index=False)
    })
    split_summary.append({
        "split": split,
        "metadata": "present",
        "audio": len(frame),
        "speakers": frame["normalized_speaker_id"].nunique(),
        "missing_audio": len(split_missing),
    })

display(pd.DataFrame(split_summary))

inventory = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if not path.is_file():
        continue
    relative = path.relative_to(OUTPUT_ROOT).as_posix()
    if relative in GENERATED_PACKAGE_FILES:
        continue
    checksum = audio_checksums.get(relative)
    if checksum is None:
        checksum = sha256_file_status(path)
    inventory.append({
        "path": relative,
        "bytes": path.stat().st_size,
        "sha256": checksum,
    })

file_manifest = pd.DataFrame(
    inventory, columns=["path", "bytes", "sha256"]
)
file_manifest.to_csv(OUTPUT_ROOT / "file_manifest.csv", index=False)

total_bytes = int(file_manifest["bytes"].sum()) if len(file_manifest) else 0
package_complete = not missing_required and not missing_audio
dataset_metadata = {
    "dataset": (
        source_manifest.get("dataset")
        if source_manifest is not None
        else "voxvietnam_ecapa_three_task_v1"
    ),
    "status": "complete" if package_complete else "incomplete",
    "root": str(OUTPUT_ROOT),
    "files": int(len(file_manifest)),
    "audio_files": int(sum(
        str(path).lower().endswith(".wav")
        for path in file_manifest["path"]
    )),
    "total_bytes": total_bytes,
    "total_gib": total_bytes / 1024**3,
    "required_files": len(required_files),
    "missing_required_files": missing_required,
    "missing_referenced_audio": missing_audio,
    "splits": split_summary,
    "source_manifest": "manifest.json" if source_manifest else None,
    "file_manifest": "file_manifest.csv",
}
(OUTPUT_ROOT / "dataset_metadata.json").write_text(
    json.dumps(dataset_metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(dataset_metadata, ensure_ascii=False, indent=2))
print("Created:", OUTPUT_ROOT / "dataset_metadata.json")
print("Created:", OUTPUT_ROOT / "file_manifest.csv")

if package_complete:
    assert source_manifest is not None
    assert source_manifest["dataset"] == "voxvietnam_ecapa_three_task_v1"
    assert source_manifest["total_audio_bytes"] <= MAX_BYTES
    assert source_manifest["invariants"]["speaker_disjoint"] is True
    assert source_manifest["invariants"]["duplicate_checksum"] == 0
    assert source_manifest["invariants"]["byte_budget_respected"] is True
    print("Package complete and ready:", OUTPUT_ROOT)
else:
    print("Package incomplete. Missing required files:", missing_required)
    print("Missing referenced audio:", missing_audio[:10])


## 11. Save to Kaggle

Do not ZIP output; ZIP temporarily duplicates disk usage.

1. Click **Save Version** and select **Save & Run All**.
2. After run completes, open notebook **Output**.
3. Create a new **private** Kaggle Dataset from folder
   `voxvietnam_ecapa_three_task_v1`.
4. Attach that dataset to
   `finetune-ecapa-for-vietnamese-speaker-indentificat.ipynb`.

Fine-tuning notebook discovers dataset through `manifest.json`; Kaggle
slug and nesting depth do not matter.
